# Module 10 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

# Forward Planner

## Unify

Use the accompanying `unification.py` file for unification. For this assignment, you're almost certainly going to want to be able to:

1. specify the problem in terms of S-expressions.
2. parse them.
3. work with the parsed versions.

`parse` and `unification` work exactly like the programming assignment for last time.

In [1]:
from unification import parse, unification, is_variable
from copy import deepcopy

## Forward Planner

In this assigment, you're going to implement a Forward Planner. What does that mean? If you look in your book, you will not find pseudocode for a forward planner. It just says "use state space search" but this is less than helpful and it's a bit more complicated than that. **(but please please do not try to implement STRIPS or GraphPlan...that is wrong).**

At a high level, a forward planner takes the current state of the world $S_0$ and attempts to derive a plan, basically by Depth First Search. We have all the ingredients we said we would need in Module 1: states, actions, a transition function and a goal test. We have a set of predicates that describe a state (and therefore all possible states), we have actions and we have, at least, an implicit transition function: applying an action in a state causes the state to change as described by the add and delete lists.

Let's say we have a drill that's an item, two places such as home and store, and we know that I'm at home and the drill is at the store and I want to go buy a drill (have it be at home). We might represent that as:

<code>
start_state = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Saw Store)",
    "(at Drill Store)",
    "(at Money Bank)"
]
</code>

And we have a goal state:

<code>
goal = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Drill Me)",
    "(at Saw Store)",
    "(at Money Bank)"
]
</code>

The actions/operators are:

<code>
actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    },
    "buy": {
        "action": "(buy ?purchaser ?seller ?item)",
        "conditions": [
            "(item ?item)",
            "(place ?seller)",
            "(agent ?purchaser)",
            "(at ?item ?seller)",
            "(at ?purchaser ?seller)"
        ],
        "add": [
            "(at ?item ?purchaser)"
        ],
        "delete": [
            "(at ?item ?seller)"
        ]
    }
}
</code>

These will all need to be parsed from s-expressions to the underlying Python representation before you can use them. You might as well do it at the start of your algorithm, once. The order of the conditions is *not* arbitrary. It is much, much better for the unification and backtracking if you have the "type" predicates (item, place, agent) before the more complex ones. Trust me on this.

As for the algorithm itself, there is going to be an *outer* level of search and an *inner* level of search.

The *outer* level of search that is exactly what I describe here: you have a state, you generate successor states by applying actions to the current state, you examine those successor states as we did at the first week of the semester and if one is the goal you stop, if you see a repeat state, you put it on the explored list (you should implement graph search not tree search). What could be simpler?

It turns out the Devil is in the details. There is an *inner* level of search hidden in "you generate successor states by applying actions to the current state". Where?

How do you know if an action applies in a state? Only if the preconditions successfully unify with the current state. That seems easy enough...you check each predicate in the conditions to see if it unifies with the current state and if it does, you use the substitution list on the action, the add and delete lists and create the successor state based on them.

Except for one small problem...there may be more than one way to unify an action with the current state. You must essentially search for all successful unifications of the candidate action and the current state. This is where my question through the semester appliesm, "how would you modify state space search to return all the paths to the goal?"

Unification can be seen as state space search by trying to unify the first precondition with the current state, progressively working your way through the precondition list. If you fail at any point, you may need to backtrack because there might have been another unification of that predicate that would succeed. Similarly, as already mentioned, there may be more than one.

So...by using unification and a properly defined <code>successors</code> function, you should be able to apply graph based search to the problem and return a "path" through the states from the initial state to the goal. You'll definitely want to use graph-based search since <code>( drive Me Store), (drive Me Home), (drive Me Store), (drive Me Home), (drive Me Store), (buy Me Store Drill), (drive Me Home)</code> is a valid plan.

Your function should return the plan...a list of actions, fully instantiated, for the agent to do in order: [a1, a2, a3]. If you pass an extra intermediate=True parameter, it should also return the resulting state of each action: [s0, a1, s1, a2, s2, a3, s3].

-----

(you can just overwrite that one and add as many others as you need). Remember to follow the **Guidelines**.


-----

So you need to implement `forward_planner` as described above. `start_state`, `goal` and `actions` should all have the layout above and be s-expressions.

Your implementation should return the plan as a **List of instantiated actions**. If `debug=True`, you should print out the intermediate states of the plan as well.

<a id="find_possible_assignment"></a>
## find_possible_assignment

When exploring the search space, it is important to find all of the possible actions that are permissible. Given a certain action, there may be multiple ways this action can be done in a certain state. This is a recursive function that identifies all the ways a specific action can occur in a specific state.

* **action_conditions** List[List]: list of the preconditions of the action to be explored
* **state** List[List]: the current state in which the action is to be explored
* **frame** List[List]: the assignments that can be made to permit the action given the current state
* **condition_index** int: counter to see which action precondition is being evaluated in specific recursion step
* **state_index** int: counter to see which state conditiont is being evaluated in specific recursion step
  
**returns** **possible_assignment** List[List]: the assignments that can be made to permit the action given the current state

In [2]:
def find_possible_assignment(action_conditions, state, frame = None, condition_index=0, state_index=0):
    if frame is None: 
        frame = {}

    if condition_index >= len(action_conditions):
            return [frame]

    frames = []
    for i in range(state_index, len(state)):
        new_frame = unification(action_conditions[condition_index], state[i], frame.copy())
        if new_frame is not False:
            extended_frames = find_possible_assignment(action_conditions, state, new_frame, condition_index + 1, 0)
            frames.extend(extended_frames)
        
    return frames

In [3]:
test_state = [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']]
test_action_conditions = [['plane', '?plane'], ['airport', '?from'], ['airport', '?to'], ['at', '?agent', '?from']]

test_actions = find_possible_assignment(test_action_conditions, test_state)

assert len(test_actions) == 2
assert type(test_actions[1]) == dict
assert len(test_actions[0]) == 4

<a id="find_possible_actions"></a>
## find_possible_actions

When exploring the search space, it is important to find all of the possible actions that are permissible. The permissibility of an action is determined by whether the preconditions of an action are satisfied by the current state. If the preconditions can be successfully unified with the current state, an action can be considered permissible. The permissible action will also have a a specific assignment, or substitution list that allows for the action to be permissible in accordance to the current state. Given a certain state, this function identifies all the ways permissible actions. 

* **state** List[List]: the current state in which the actions are to be explored
* **actions** Dict[str, Dict]: dictionary of all possible actions, their preconditions, add list and delete list

**returns** **possible_actions** List[Tuple]: list of  permissible actions, and corresponding substitution list in accordance to the current state

In [4]:
def find_possible_actions(state, actions):
    possible_actions = {}

    for action_name, action in actions.items():
        possible_assignment = find_possible_assignment(action['conditions'], state)

        if possible_assignment: 
            possible_actions[action_name]= possible_assignment

    possible_actions_list = []
    for action, assignments in possible_actions.items():
        for assignment in assignments:
            possible_actions_list.append((action, assignment))
        
    return possible_actions_list

In [5]:
test_state = [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']]
test_actions = {'fly': {'action': ['fly', '?plane', '?from', '?to'], 'conditions': [['plane', '?plane'], ['airport', '?from'], ['airport', '?to'], ['at', '?plane', '?from']], 'add': [['at', '?plane', '?to']], 'delete': [['at', '?plane', '?from']]}}

test_possible_actions = find_possible_actions(test_state, test_actions)
assert len(test_possible_actions) == 2
assert test_possible_actions[0] == ('fly', {'?plane': '1973', '?from': 'SFO', '?to': 'SFO'})
assert test_possible_actions[1] == ('fly', {'?plane': '1973', '?from': 'SFO', '?to': 'JFK'})

<a id="apply_substitution"></a>
## apply_substitution

In order to continue unification of two expressions, the assignments made should be reflected in the remainder of the string. In doing so, the assignments can be propagated through the unification of the string if a variable occurs more than once. This also prevents different values/variables from being assigned to the same variable. This function adjusts the expression with the assignments made in the substitution list. 

* **substitution_list** dict: dictionary of the substitutions found and assignments made
* **expression** list[str]: expression to which the substitutions should be applied 
  
**returns** **expression** expression after substitutions were made

In [6]:
def apply_substitution(substitution_list, expression):
    if is_variable(expression):
        return substitution_list.get(expression, expression)
    elif isinstance(expression, list):
        return [apply_substitution(substitution_list, item) for item in expression]
    else:
        return expression

In [7]:
assert apply_substitution({'?x':'?y'}, ['?x', '?x', '?y']) == ['?y', '?y', '?y']

assert apply_substitution({'?x':'?y', '?y': '3'}, ['?x', '?x', '?y']) == ['?y', '?y', '3']

assert apply_substitution({'?x':'?y'}, ['Barney', ['Fred', '?x']]) == ['Barney', ['Fred', '?y']]

<a id="apply_action"></a>
## apply_action

In order to identify a plan of action to get from the start state to the goal state, each permissible action has to be applied to the current state sequentially to obtain the new state. The new state is then evaluated for further permissible actions. Changes to the state can dictate the trajecotry of the path there after. This function applies the changes to the state when a permissible action is taken. The function removes the state conditions as listed in the delete list, and adds the state conditions as listed in the add list.  

* **action** dict: dictionary containing the preconditions, add list and delete list of the applied action
* **assignment** dict: substitution list that details the action be taken
* **state** List[List]: current state to which the action is being applied
  
**returns** **new_state** List[List]: new state after an action has been taken

In [8]:
def apply_action(action, assignment, state):
    new_state = deepcopy(state)
    for deletion in action['delete']:
        delete = apply_substitution(assignment, deletion)
        new_state.remove(delete)
    for addition in action['add']:
        add = apply_substitution(assignment, addition)
        new_state.append(add)
    return new_state

In [9]:
test_action = {'action': ['fly', '?plane', '?from', '?to'], 'conditions': [['plane', '?plane'], ['airport', '?from'], ['airport', '?to'], ['at', '?plane', '?from']], 'add': [['at', '?plane', '?to']], 'delete': [['at', '?plane', '?from']]}
test_assignment = {'?plane': '1973', '?from': 'SFO', '?to': 'JFK'}
test_state = [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']]

test_new_state = apply_action(test_action, test_assignment, test_state)

assert len(test_new_state) == 4
assert ['at', '1973', 'JFK'] in test_new_state
assert ['at', '1973', 'SFO'] not in test_new_state

<a id="dfs"></a>
## dfs

Forward planning is an AI method used to generate a path to get from an initial state to a goal state. Depth first search (DFS) is a method used to explore the state space and identify potential actions that can be taken. The depth-first method traversing along a single path in its entirety until the goal state is reached, or backtracking is required to explore alternative paths. This function applies depth first search in a recursive manner to identify a plan of actions to get from the initial state to the goal state, given a certain set of possible actions. 

* **state** List[List]: current state
* **goal** List[List]: goal state
* **actions** Dict[str, Dict]: dictionary of all possible actions, their preconditions, add list and delete list
* **plan** List: List of successful actions taken so far in path
* **visited** Set: Set of states already explored along this path
* **debug** Boolean: used to print output to help in debugging the program
  
**returns** **plan** List[List]: List of successful actions to be taken in current path/plan

In [10]:
def dfs(state, goal, actions, plan=[], visited=set(), debug=False):
    if debug:
        print(f"Current State: {state}")
        print(f"Current Plan: {plan}\n")
        
    #base cases
    if all(item in state for item in goal):
        if debug:
            print("Goal reached")
        return plan

    sortable_state = sorted(map(tuple, state))
    tuple_state = tuple(sortable_state)
    
    if tuple_state in visited:
        return None

    visited_deepcopy = deepcopy(visited)
    visited_deepcopy.add(tuple_state)

    possible_actions = find_possible_actions(state, actions)

    for action_name, assignment in possible_actions:
        new_state = apply_action(actions[action_name], assignment, state)
        new_plan = deepcopy(plan) + [apply_substitution(assignment, actions[action_name]['action'])]
        result = dfs(new_state, goal, actions, new_plan, visited_deepcopy, debug=debug)
        if result is not None:
            return result
        
    return None

In [11]:
test_state = [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']]
test_goal = [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'JFK']]
test_actions = {'fly': {'action': ['fly', '?plane', '?from', '?to'], 'conditions': [['plane', '?plane'], ['airport', '?from'], ['airport', '?to'], ['at', '?plane', '?from']], 'add': [['at', '?plane', '?to']], 'delete': [['at', '?plane', '?from']]}}

print(dfs(test_state, test_goal, test_actions))

[['fly', '1973', 'SFO', 'JFK']]


<a id="parse_everything"></a>
## parse_everything

Before applying the depth-first-search on the current state using the goal state and the actions, the input data needs to be formatted to support the Python representations. The s-expressions found in the input need to be parsed into the underlying Python representation. This function parses the start state, goal and actions into the required format. 

* **state** List[Str]: current state
* **goal** List[Str]: goal state
* **actions** Dict[str, Dict]: dictionary of all possible actions, their preconditions, add list and delete list
  
**returns** **state, goal, actions** List[List], List[List], Dict[str: List]: Parsed versions of the input 

In [12]:
def parse_everything(start_state, goal, actions):
    parsed_start_state = [parse(s) for s in start_state]
    parsed_goal = [parse(g) for g in goal]
    parsed_actions = {
        action_name: {
            'action': parse(action['action']),
            'conditions': [parse(cond) for cond in action['conditions']],
            'add': [parse(add) for add in action['add']],
            'delete': [parse(delete) for delete in action['delete']],
        } for action_name, action in actions.items()
    }
    return parsed_start_state, parsed_goal, parsed_actions

In [13]:
test_start_state = ["(item Saw)", "(item Drill)"]
test_goal = ["(item Saw)", "(item Drill)"]
test_actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    }
}
parsed_start_state, parsed_goal, parsed_actions = parse_everything(test_start_state, test_goal, test_actions)

assert parsed_start_state == [['item', 'Saw'], ['item', 'Drill']]
assert parsed_goal == [['item', 'Saw'], ['item', 'Drill']]
assert parsed_actions == {'drive': {'action': ['drive', '?agent', '?from', '?to'], 'conditions': [['agent', '?agent'], ['place', '?from'], ['place', '?to'], ['at', '?agent', '?from']], 'add': [['at', '?agent', '?to']], 'delete': [['at', '?agent', '?from']]}}

<a id="forward_planner"></a>
## forward_planner

Forward planning is an AI method used to generate a path to get from an initial state to a goal state. A specific set of possible actions are used to navigate the state space, allowing for transition from one state to the next. Each action has specific preconditions that need to be satisfied before the action takes place. As the action takes place, additions and deletions are made to the state space to signify a new state. This occurs until the goal state is reached. Depth first search (DFS) is a method used to explore the state space and identify potential actions that can be taken in a recursive manner. 

This function parses the input data into a Python-supported format and initiates the recursive depth first search of the state space to identify a permissible plan of action to get from the current state to the goal state. 

* **state** List[Str]: current state
* **goal** List[Str]: goal state
* **actions** Dict[str, Dict]: dictionary of all possible actions, their preconditions, add list and delete list
* **debug** Boolean: used to print output to help in debugging the program
  
**returns** **state, goal, actions** List[List], List[List], Dict[str: List]: Parsed versions of the input 

In [14]:
def forward_planner(start_state, goal, actions, debug=False):
    start_state, goal, actions = parse_everything(start_state, goal, actions)

    if debug:
        print("Starting State:", start_state,'\n')
        print("Goal State:", goal,'\n')
        print("Actions:", actions,'\n')
    
    plan = dfs(start_state, goal, actions, debug=debug)
    return plan

In [15]:
test_state = [
    "(plane 1973)",
    "(airport SFO)",
    "(airport JFK)",
    "(at 1973 SFO)"
]

test_goal = [
    "(plane 1973)",
    "(airport SFO)",
    "(airport JFK)",
    "(at 1973 JFK)",
]

test_actions = {
    "fly": {
        "action": "(fly ?plane ?from ?to)",
        "conditions": [
            "(plane ?plane)",
            "(airport ?from)",
            "(airport ?to)",
            "(at ?plane ?from)"
        ],
        "add": [
            "(at ?plane ?to)"
        ],
        "delete": [
            "(at ?plane ?from)"
        ]
    }
}

test_plan = forward_planner(test_state, test_goal, test_actions, debug=True)
assert len(test_plan) == 1
assert len(test_plan[0]) == 4
assert test_plan == [['fly', '1973', 'SFO', 'JFK']]

Starting State: [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']] 

Goal State: [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'JFK']] 

Actions: {'fly': {'action': ['fly', '?plane', '?from', '?to'], 'conditions': [['plane', '?plane'], ['airport', '?from'], ['airport', '?to'], ['at', '?plane', '?from']], 'add': [['at', '?plane', '?to']], 'delete': [['at', '?plane', '?from']]}} 

Current State: [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']]
Current Plan: []

Current State: [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'SFO']]
Current Plan: [['fly', '1973', 'SFO', 'SFO']]

Current State: [['plane', '1973'], ['airport', 'SFO'], ['airport', 'JFK'], ['at', '1973', 'JFK']]
Current Plan: [['fly', '1973', 'SFO', 'JFK']]

Goal reached


You will be solving the problem from above. Here is the start state:

In [16]:
start_state = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Saw Store)",
    "(at Drill Store)"
]

The goal state:

In [17]:
goal = [
    "(item Saw)",    
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",    
    "(agent Me)",
    "(at Me Home)",
    "(at Drill Me)",
    "(at Saw Store)"    
]

and the actions/operators:

In [18]:
actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    },
    "buy": {
        "action": "(buy ?purchaser ?seller ?item)",
        "conditions": [
            "(item ?item)",
            "(place ?seller)",
            "(agent ?purchaser)",
            "(at ?item ?seller)",
            "(at ?purchaser ?seller)"
        ],
        "add": [
            "(at ?item ?purchaser)"
        ],
        "delete": [
            "(at ?item ?seller)"
        ]
    }
}

**Note** The facts for each state are really an ordered set. When comparing two states, you may need to convert them to a Set first.

In [19]:
plan = forward_planner(start_state, goal, actions, False)

In [20]:
for el in plan:
    print(el)

['drive', 'Me', 'Home', 'Store']
['buy', 'Me', 'Store', 'Drill']
['drive', 'Me', 'Store', 'Home']


## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.